In [1]:
# Step 1: Install Dependencies
!pip install llama-index llama-index-vector-stores-chroma llama-index-embeddings-huggingface llama-index-llms-huggingface
!pip install chromadb
# !pip install unsloth
!pip install transformers accelerate bitsandbytes
!pip install sentence-transformers
!pip install python-docx

print("All dependencies installed successfully!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.6 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of llama-cloud-services to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of llama-cloud-services to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 82.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 85.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.3/303.3 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━

In [2]:
# Step 2: Authentication & Configuration
from huggingface_hub import login


HF_TOKEN = ""  
login(token=HF_TOKEN)

MODEL_ID = "Amr20004/GEM-TourGuide-Gemma3n-v3"
EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
CHROMA_COLLECTION_NAME = "gem_knowledge_base"
TOP_K = 5  # Number of relevant chunks to retrieve
MAX_SEQ_LENGTH = 2048

print("Configuration set!")
print(f"LLM: {MODEL_ID}")
print(f"Embeddings: {EMBEDDING_MODEL}")
print(f"Top-K retrieval: {TOP_K}")

Configuration set!
LLM: Amr20004/GEM-TourGuide-Gemma3n-v3
Embeddings: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Top-K retrieval: 5


In [5]:
# Step 3: Loading the data
from docx import Document
import re

document = Document('/kaggle/input/datasets/amr2004/final-artifacts-dataset/artifacts.docx')
raw_text = ""

for paragraph in document.paragraphs:
    raw_text += paragraph.text + "\n"

# print(raw_text)
chunks = re.findall(r'--(.+?)--', raw_text, re.DOTALL)
print(chunks[0])

Statue Head of King Akhenaten
This head is from one of several colossal statues that were originally placed in front of a temple for the sun god, Aten. They were carved early in the reign of King Akhenaten. His narrow almond-shaped eyes and long face show the Amarna artistic style, which was unique to this time in Egyptian history. On top of the nemes headcloth are four carved ostrich feathers, which showed Akhenaten's connection with Shu, the god of the air. New Kingdom, Dynasty 18, reign of Amenhotep IV/ Akhenaten, about 1353 - 1336 BC | Sandstone | From the Aten Colonnade at Karnak, Thebes (Luxor).


In [6]:
# Step 4: create the documents
from llama_index.core import Document

documents = []
artifact_titles = []
for chunk in chunks:
    lines = chunk.strip().split('\n', 1)
    title = lines[0].strip()
    description = lines[1].strip() if len(lines) > 1 else ""

    doc = Document(
        text = chunk.strip(),
        metadata = {"title":title}
    )
    documents.append(doc)
    artifact_titles.append(title)

# print(documents[0])
print(f"there are {len(documents)} documents")
print(f"there are {len(artifact_titles)} Titles/Ids")
print(documents[0])
print(documents[9])

there are 107 documents
there are 107 Titles/Ids
Doc ID: d6f60bbd-0ad3-4fbe-8a18-8979523b8f4f
Text: Statue Head of King Akhenaten This head is from one of several
colossal statues that were originally placed in front of a temple for
the sun god, Aten. They were carved early in the reign of King
Akhenaten. His narrow almond-shaped eyes and long face show the Amarna
artistic style, which was unique to this time in Egyptian history. On
top of the...
Doc ID: 41ddb5de-ddd8-445b-b60d-75c752d36f10
Text: جزء من تمثال ملكي لاتزال التفاصيل المُلَوَّنة واضحة على:
العينين، والحاجبين، وغطاء الرأس (النمس)، المنحوت بدقة عالية، وعلى
الرغم من عدم ذكر اسم صاحب هذا التمثال، إلا أن ملامح الوجه، تُشِير إلى
الملك أمنحتب الثالث. الأسرة 18  ألباستر مصري، أكاسيد لونية الكرنك


In [7]:
# Step 4 indexing the documents
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import StorageContext, VectorStoreIndex
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# 1. Initialize embedding model
print("Loading embedding model...")
embed_model = HuggingFaceEmbedding(model_name=EMBEDDING_MODEL, device="cuda:1")
print(f"Embedding model loaded: {EMBEDDING_MODEL}")

# 2. Initialize ChromaDB
print("\n Initializing ChromaDB...")
chroma_client = chromadb.PersistentClient(path="./chroma_db")  # Persistent storage

# Delete existing collection if re-running
try:
    chroma_client.delete_collection(CHROMA_COLLECTION_NAME)
    print("   (Deleted existing collection for fresh start)")
except:
    pass

chroma_collection = chroma_client.create_collection(
    name=CHROMA_COLLECTION_NAME,
    metadata={"description": "GEM Museum Knowledge Base for Egyptian Cultural Heritage AI"}
)
print(f"ChromaDB collection created: {CHROMA_COLLECTION_NAME}")

# 3. Create LlamaIndex vector store
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

# 4. Build the index 
print("\n Embedding and indexing all documents (this may take a minute)...")
index = VectorStoreIndex.from_documents(
    documents,
    storage_context=storage_context,
    embed_model=embed_model,
    show_progress=True
)

print(f"\n Vector store built successfully!")
print(f"   Total documents indexed: {chroma_collection.count()}")

2026-02-26 18:15:16.161384: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772129716.370006      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772129716.430640      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772129716.930375      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772129716.930413      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772129716.930416      55 computation_placer.cc:177] computation placer alr

Loading embedding model...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2

 Initializing ChromaDB...
ChromaDB collection created: gem_knowledge_base

 Embedding and indexing all documents (this may take a minute)...


Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/107 [00:00<?, ?it/s]


 Vector store built successfully!
   Total documents indexed: 107


In [8]:
# Step 5: Loading the fine-tuned Gemma3n model
import torch
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer

gc.collect()
torch.cuda.empty_cache()

# MODEL_ID = "Amr20004/Egyptology-Structured-Gemma3n"
MODEL_ID = "Amr20004/GEM-TourGuide-Gemma3n-v3"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Load in float16 to CPU first 
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="cpu",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
)

print(" Loaded to CPU, moving to GPU...")

# Now move to GPU 
model = model.to("cuda:0")

print(f" Model on GPU!")
print(f"   VRAM used: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.70M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

...

model-00001-of-00003.safetensors:   0%|          | 0.00/3.72G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/210 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/154M [00:00<?, ?B/s]

 Loaded to CPU, moving to GPU...
 Model on GPU!
   VRAM used: 9.33 GB


In [7]:
# Step 6: Build the RAG Query Engine
import json
import re

# === RAG System Prompt ===
SYSTEM_PROMPT = """You are an expert Egyptology AI assistant serving as a tour guide at the 
Grand Egyptian Museum (GEM) in Giza, Egypt. You help tourists understand ancient Egyptian 
artifacts, pharaohs, and history.

IMPORTANT RULES:
1. ONLY use information from the provided context to answer questions.
2. If the context doesn't contain enough information, say "I don't have specific information 
   about that in my knowledge base" rather than making up facts.
3. Be engaging and educational, like a knowledgeable museum guide.
4. When discussing artifacts, mention their materials, dimensions, and historical significance.
5. When discussing pharaohs, include their dynasty, reign dates, and key achievements.
6. Provide the answer as structured JSON.
"""

def detect_language(text: str) -> str:
    arabic_chars = sum(1 for c in text if '\u0600' <= c <= '\u06FF')
    return "arabic" if arabic_chars > len(text) * 0.3 else "english"


def build_rag_prompt(query: str, retrieved_contexts: list) -> str:
    """Build the prompt that combines retrieved context with the user's question."""

    language = detect_language(query)

    if language == "arabic":
        language_instruction = "CRITICAL: The tourist is speaking Arabic. You MUST respond ENTIRELY in Arabic. All JSON values must be in Arabic."
    else:
        language_instruction = "The tourist is speaking English. Respond in English."
    
    # Format retrieved contexts
    context_parts = []
    for i, node in enumerate(retrieved_contexts, 1):
        context_parts.append(
            f"---\n{node.text}\n"
        )
    
    context_block = "\n".join(context_parts)
    
    prompt = (
        f"{SYSTEM_PROMPT}\n\n"
        f"{language_instruction}\n\n"
        f"=== RETRIEVED KNOWLEDGE BASE CONTEXT ===\n"
        f"{context_block}\n"
        f"=== END OF CONTEXT ===\n\n"
        f"Tourist's Question: {query}\n\n"
        f"Please provide the answer as structured JSON."
    )
    
    return prompt


def generate_response(prompt: str, max_new_tokens: int = 1024) -> str:
    """Generate a response from the fine-tuned Gemma3n model."""
    
    messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
    
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        tokenize=True,
        return_dict=True
    ).to("cuda")
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.2,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.1,
        )
    
    generated_ids = outputs[0][inputs['input_ids'].shape[1]:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    
    return response



def parse_json_response(response: str):
    """Parse JSON from model response with robust error fixing for LLM outputs."""
    # Step 1: Clean markdown fences
    clean = response.replace("```json", "").replace("```", "").strip()
    
    # Step 2: Extract JSON block if there's extra text around it
    start = clean.find("{")
    end = clean.rfind("}") + 1
    if start != -1 and end > start:
        clean = clean[start:end]
    else:
        return {"raw_response": response}
    
    # Step 3: Try direct parse first
    try:
        return json.loads(clean)
    except json.JSONDecodeError:
        pass
    
    # Step 4: Fix common LLM JSON errors line by line
    # The most common error: missing opening quote on values
    # e.g.  "key": value"  instead of  "key": "value"
    lines = clean.split('\n')
    fixed_lines = []
    for line in lines:
        # Match:  "key": unquoted_value"  (missing opening quote)
        match = re.match(r'(\s*"[^"]*"\s*:\s*)([^"\[{\d\s][^"\n]*")(.*)', line)
        if match:
            prefix = match.group(1)     # e.g. '  "discovery_location": '
            bad_val = match.group(2)    # e.g. 'معبدها الجنائزي بالدير البحري"'
            suffix = match.group(3)     # e.g. ','
            line = prefix + '"' + bad_val + suffix
        fixed_lines.append(line)
    fixed = '\n'.join(fixed_lines)
    
    # Fix trailing commas before } or ]
    fixed = re.sub(r',\s*([}\]])', r'\1', fixed)
    
    try:
        return json.loads(fixed)
    except json.JSONDecodeError:
        pass
    
    # Step 5: Extract all properly quoted key-value pairs manually
    try:
        result = {}
        # Extract "key": "value" pairs
        str_pairs = re.findall(r'"([^"]+)"\s*:\s*"((?:[^"\\]|\\.)*)"', clean)
        for key, val in str_pairs:
            result[key] = val
        # Extract "key": number pairs
        num_pairs = re.findall(r'"([^"]+)"\s*:\s*(\d+(?:\.\d+)?)\s*[,}\n]', clean)
        for key, val in num_pairs:
            result[key] = int(val) if '.' not in val else float(val)
        # Extract "key": null pairs
        null_pairs = re.findall(r'"([^"]+)"\s*:\s*null', clean)
        for key in null_pairs:
            result[key] = None
        if result:
            return result
    except:
        pass
    
    return {"raw_response": response}


print("RAG query functions defined!")

RAG query functions defined!


In [8]:
# Step 7: The Main RAG Query Function
from llama_index.core.schema import TextNode, NodeWithScore

def retrieve_by_artifact_id(artifact_id: str, top_k: int = 3):
    try:
        results = chroma_collection.get(where={"title":artifact_id}, include=["documents","metadatas"])
        if results and results["documents"]:
            retrieved_nodes = []
            for doc_text, metadata in zip(results["documents"], results["metadatas"]):
                node = TextNode(text=doc_text, metadata=metadata)
                nodeWithScore = NodeWithScore(node=node, score=1.0)
                retrieved_nodes.append(nodeWithScore)
            return retrieved_nodes[:top_k]
    except Exception as e:
        print(f"  Metadata filter failed: {e}, falling back to similarity search")

    # Fallback use the artifact_id as a similarity query
    retriever = index.as_retriever(similarity_top_k=top_k)
    return retriever.retrieve(artifact_id)
    


def ask_gem_guide(question: str, artifact_id: str = None, top_k: int = TOP_K, verbose: bool = True) -> dict:
    """
    Main RAG function — the tourist asks a question, and the AI guide answers
    using retrieved knowledge from the GEM database.
    
    Args:
        question: The tourist's question (text)
        top_k: Number of relevant documents to retrieve
        verbose: Whether to print detailed output
    
    Returns:
        dict with 'answer' (parsed JSON), 'raw_response', and 'sources'
    """
    mode = "camera" if artifact_id else "text"
    
    
    if verbose:
        print(f"\n{'='*60}")
        if mode == "camera":
            print(f" Mode: CAMERA (artifact identified)")
            print(f" Artifact ID: {artifact_id}")
            print(f" Tourist's Question: {question}")
        else:
            print(f" Mode: TEXT (no camera)")
            print(f" Tourist's Question: {question}")
        print(f"{'='*60}")

    if mode == "camera":
        retrieved_nodes = retrieve_by_artifact_id(artifact_id)   
        print(f"\n Retrieved {len(retrieved_nodes)} chunks (by artifact ID):")
        for i, node in enumerate(retrieved_nodes, 1):
            title = node.metadata.get('title', 'N/A')[:50]
            print(f"   {i}. {title} — Score: {node.score:.4f}")
    
    else:
        retriever = index.as_retriever(similarity_top_k=top_k)
        retrieved_nodes = retriever.retrieve(question)
        if verbose:
            print(f"\n Retrieved {len(retrieved_nodes)} chunks:")
            for i, node in enumerate(retrieved_nodes, 1):
                print(f"   {i}. {node.metadata.get('title', 'N/A')[:50]} — Score: {node.score:.4f}")

    
    # Step 2: Build the RAG prompt
    prompt = build_rag_prompt(question, retrieved_nodes)
    if verbose:
        print(f"\n Prompt length: {len(prompt)} characters")

    
    # Step 3: Generate response
    if verbose:
        print(f"\n Generating response...")
    raw_response = generate_response(prompt)

    
    # Step 4: Parse response
    parsed = parse_json_response(raw_response)
    if verbose:
        print(f"\n AI Guide Response:")
        print(json.dumps(parsed, indent=2, ensure_ascii=False))


    
    # Step 5: Return structured result
    sources = [
        {
            "title": node.metadata.get('title', 'N/A'),
            "score": float(node.score),
        }
        for node in retrieved_nodes
    ]
    
    return {
        "question": question,
        "artifact_id": artifact_id,
        "mode": mode,
        "answer": parsed,
        "raw_response": raw_response,
        "sources": sources,
    }


print(" RAG query engine ready!")

 RAG query engine ready!


In [10]:
# Test 1
result1 = ask_gem_guide(
    question = "tell me about this artifact",
    artifact_id = "Seated Statue of King Amenhotep III"
)



 Mode: CAMERA (artifact identified)
 Artifact ID: Seated Statue of King Amenhotep III
 Tourist's Question: tell me about this artifact

 Retrieved 1 chunks (by artifact ID):
   1. Seated Statue of King Amenhotep III — Score: 1.0000

 Prompt length: 2001 characters

 Generating response...

 AI Guide Response:
{
  "artifact_name": "Seated Statue of King Amenhotep III",
  "description": "This seated statue depicts King Amenhotep III, one of Egypt's most powerful and wealthiest rulers of Dynasty 18. During his reign (circa 1390-1352 BC), he oversaw extensive building projects including temples and palaces across Egypt. The sheer number of statues of him found dates to his reign – over 1000 life-sized or larger! This particular statue may have been created to commemorate one of his royal jubileums.",
  "material": "Granite",
  "provenance": "Mortuary Cult Temple of Amenhotep III, Kom el-Hetan, Thebes (Luxor)",
  "dynasty": "18th Dynasty",
  "reign": "circa 1390-1352 BC",
  "artistic_signi

In [9]:
# Test 2
result2 = ask_gem_guide(
    question = "اخبرني عن هذا التمثال",
    artifact_id = "Ramesses II Seated Statue"
)


 Mode: CAMERA (artifact identified)
 Artifact ID: Ramesses II Seated Statue
 Tourist's Question: اخبرني عن هذا التمثال

 Retrieved 1 chunks (by artifact ID):
   1. Ramesses II Seated Statue — Score: 1.0000

 Prompt length: 1884 characters

 Generating response...

 AI Guide Response:
{
  "العنوان": "تمثال الملك رمسيس الثاني الجالس",
  "الوصف بالعربية": "هذا التمثال هو جزء من مجموعة تماثيل للملك رمسيس الثاني يعتقد أنها تعود إلى بداية حكمه، وقد عُثر عليها في شرق الدلتا. النقوش تصفه بـ«حاكم الأرضين وحاكم السعادة». التمثال مصنوع من جرانيت بلون الشمس ويأتي من تانيس (صان الحجر)، وهي منطقة في شرق الدلتا المصرية. الملك رمسيس الثاني عاش فترة استقرار داخلي وخارجي طويلة استمرت ستة وستون عامًا، وشهدت ظهور تماثيل جديدة ومعابد ومدن جديدة في جميع أنحاء الإمبراطورية.",
  "الأسرة": "الأسرة التاسعة عشرة",
  "المادة": "جرانيت بلون الشمس",
  "الموقع الاكتشافي": "شرق الدلتا - تانيس (صان الحجر)",
  "الأهمية التاريخية": "يشير وجود هذه التماثيل إلى فترة استقرار وازدهار شهدها عهد الملك رمسيس الثاني الذي حكم م